# Notebook 02: Base Market Feature Engineering

## Project
NEM Network Constraints & Price Divergence Intelligence System

## Objective
Create the first layer of market features from the clean NSW1 and VIC1 base market context table.

## Business Question
Before introducing constraints and interconnector flows, what does normal market behaviour tell us about price spikes, volatility, demand conditions, and regional tightness?


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)


In [2]:
PROJECT_ROOT = (
    Path.cwd().parents[0]
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"

BASE_MARKET_FILE = OUTPUT_DIR / "01_base_market_context.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", BASE_MARKET_FILE)


Project root: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence
Input file: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/01_base_market_context.csv


In [3]:
market_features = pd.read_csv(
    BASE_MARKET_FILE,
    parse_dates=["settlementdate"]
)

market_features.head()


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0


In [4]:
print("Shape:", market_features.shape)
print("Date range:", market_features["settlementdate"].min(), "to", market_features["settlementdate"].max())
print("Regions:", market_features["regionid"].unique())


Shape: (16126, 8)
Date range: 2026-02-01 00:05:00 to 2026-02-28 23:55:00
Regions: ['NSW1' 'VIC1']


In [5]:
market_features.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16126 entries, 0 to 16125
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   settlementdate          16126 non-null  datetime64[ns]
 1   regionid                16126 non-null  object        
 2   rrp                     16126 non-null  float64       
 3   intervention            16126 non-null  int64         
 4   totaldemand             16126 non-null  float64       
 5   availablegeneration     16126 non-null  float64       
 6   netinterchange          16126 non-null  float64       
 7   intervention_regionsum  16126 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(2), object(1)
memory usage: 1008.0+ KB


## Step 6: Supply-Demand Gap

### What we are doing
We are creating a simple market tightness feature by subtracting regional demand from available generation.

### Why it matters
A smaller supply-demand gap can indicate tighter regional conditions. Tight conditions may increase price sensitivity, especially when constraints or interconnector limits are present.

### Formula
supply_demand_gap = availablegeneration - totaldemand


In [6]:
market_features["supply_demand_gap"] = (
    market_features["availablegeneration"] - market_features["totaldemand"]
)

market_features[
    [
        "settlementdate",
        "regionid",
        "totaldemand",
        "availablegeneration",
        "supply_demand_gap",
    ]
].head()


,settlementdate,regionid,totaldemand,availablegeneration,supply_demand_gap
0,2026-02-01 00:05:00,NSW1,8186.33,13272.53373,5086.20373
1,2026-02-01 00:10:00,NSW1,8165.29,13246.75420,5081.46420
2,2026-02-01 00:15:00,NSW1,8202.70,13198.51267,4995.81267
3,2026-02-01 00:20:00,NSW1,8213.41,13169.71135,4956.30135
4,2026-02-01 00:25:00,NSW1,8055.53,13139.84760,5084.31760


## Step 7: Price Spike Flag

### What we are doing
We are creating a flag for dispatch intervals where the regional reference price is above $300/MWh.

### Why it matters
This identifies price events that require further explanation. At this stage, we are only flagging spikes, not explaining whether they were caused by demand, volatility, constraints, or interconnector congestion.

### Rule
price_spike_flag = True when rrp > 300


In [8]:
PRICE_SPIKE_THRESHOLD = 300

market_features["price_spike_flag"] = (
    market_features["rrp"] > PRICE_SPIKE_THRESHOLD
)

market_features[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "price_spike_flag",
    ]
].head()


,settlementdate,regionid,rrp,price_spike_flag
0,2026-02-01 00:05:00,NSW1,57.05984,False
1,2026-02-01 00:10:00,NSW1,64.51979,False
2,2026-02-01 00:15:00,NSW1,65.08727,False
3,2026-02-01 00:20:00,NSW1,64.89000,False
4,2026-02-01 00:25:00,NSW1,63.47180,False


## Step 8: Price Change

### What we are doing
We are calculating t
zhe change in RRP from the previous 5-minute dispatch interval for each region.

### Why it matters
Large interval-to-interval price changes are an early signal of volatility. They can indicate sudden changes in supply, demand, bidding conditions, constraints, or interconnector availability.

### Formula
price_change = current interval RRP - previous interval RRP


In [9]:
market_features = market_features.sort_values(
    ["regionid", "settlementdate"]
).reset_index(drop=True)

market_features["price_change"] = (
    market_features
    .groupby("regionid")["rrp"]
    .diff()
)

market_features[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "price_change",
    ]
].head(10)


,settlementdate,regionid,rrp,price_change
0,2026-02-01 00:05:00,NSW1,57.05984,NaN
1,2026-02-01 00:10:00,NSW1,64.51979,7.45995
2,2026-02-01 00:15:00,NSW1,65.08727,0.56748
3,2026-02-01 00:20:00,NSW1,64.89000,-0.19727
4,2026-02-01 00:25:00,NSW1,63.47180,-1.41820
5,2026-02-01 00:30:00,NSW1,64.85825,1.38645
6,2026-02-01 00:35:00,NSW1,64.05037,-0.80788
7,2026-02-01 00:40:00,NSW1,64.89000,0.83963
8,2026-02-01 00:45:00,NSW1,64.14536,-0.74464
9,2026-02-01 00:50:00,NSW1,62.95677,-1.18859


## Step 9: Rolling One-Hour Price Volatility

### What we are doing
We are calculating rolling price volatility using the standard deviation of RRP over the previous 12 dispatch intervals.

### Why it matters
The NEM dispatch interval is 5 minutes. Twelve dispatch intervals represent one hour. A high rolling standard deviation means prices have been unstable over the recent trading period.

### Formula
rolling_rrp_volatility_1h = standard deviation of RRP over previous 12 dispatch intervals


In [10]:
ROLLING_WINDOW_INTERVALS = 12

market_features["rolling_rrp_volatility_1h"] = (
    market_features
    .groupby("regionid")["rrp"]
    .transform(
        lambda x: x.rolling(
            window=ROLLING_WINDOW_INTERVALS,
            min_periods=3
        ).std()
    )
)

market_features[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "rolling_rrp_volatility_1h",
    ]
].head(15)


,settlementdate,regionid,rrp,rolling_rrp_volatility_1h
0,2026-02-01 00:05:00,NSW1,57.05984,NaN
1,2026-02-01 00:10:00,NSW1,64.51979,NaN
2,2026-02-01 00:15:00,NSW1,65.08727,4.479816
3,2026-02-01 00:20:00,NSW1,64.89000,3.893369
4,2026-02-01 00:25:00,NSW1,63.47180,3.381808
5,2026-02-01 00:30:00,NSW1,64.85825,3.117894
6,2026-02-01 00:35:00,NSW1,64.05037,2.859792
7,2026-02-01 00:40:00,NSW1,64.89000,2.698207
8,2026-02-01 00:45:00,NSW1,64.14536,2.530398
9,2026-02-01 00:50:00,NSW1,62.95677,2.396131


## Step 10: Time Features

### What we are doing
We are extracting calendar and time-of-day features from the dispatch interval timestamp.

### Why it matters
Electricity price behaviour is strongly time-dependent. Morning peaks, evening peaks, weekday 
demand, and weekend demand can all produce different price patterns.

### Features created
- date
- hour
- weekday
- is_weekend


In [12]:
market_features["date"] = market_features["settlementdate"].dt.date
market_features["hour"] = market_features["settlementdate"].dt.hour
market_features["weekday"] = market_features["settlementdate"].dt.day_name()
market_features["is_weekend"] = market_features["settlementdate"].dt.weekday >= 5

market_features[
    [
        "settlementdate",
        "regionid",
        "date",
        "hour",
        "weekday",
        "is_weekend",
    ]
].head()


,settlementdate,regionid,date,hour,weekday,is_weekend
0,2026-02-01 00:05:00,NSW1,2026-02-01,0,Sunday,True
1,2026-02-01 00:10:00,NSW1,2026-02-01,0,Sunday,True
2,2026-02-01 00:15:00,NSW1,2026-02-01,0,Sunday,True
3,2026-02-01 00:20:00,NSW1,2026-02-01,0,Sunday,True
4,2026-02-01 00:25:00,NSW1,2026-02-01,0,Sunday,True


## Step 11: Validate Feature Missing Values

### What we are doing
We are checking missing values after feature engineering.

### Why it matters
Some missing values are expected. The first price change for each region is missing because there is no previous dispatch interval. Rolling volatility is also missing at the start of each region because the rolling window needs previous intervals.

### Expected missing values
- price_change: first row for each region
- rolling_rrp_volatility_1h: first few rows for each region


In [14]:
feature_missing_values = market_features[
    [
        "rrp",
        "totaldemand",
        "availablegeneration",
        "netinterchange",
        "supply_demand_gap",
        "price_spike_flag",
        "price_change",
        "rolling_rrp_volatility_1h",
    ]
].isna().sum()

feature_missing_values


rrp                          0
totaldemand                  0
availablegeneration          0
netinterchange               0
supply_demand_gap            0
price_spike_flag             0
price_change                 2
rolling_rrp_volatility_1h    4
dtype: int64

## Step 12: Create Regional Feature Summary

### What we are doing
We are summarising the engineered market features by region.

### Why it matters
This gives a first comparison between NSW1 and VIC1 before network effects are introduced. We can see which region had higher prices, more spike intervals, stronger demand, and more volatile price behaviour.


In [16]:
regional_feature_summary = (
    market_features
    .groupby("regionid")
    .agg(
        intervals=("settlementdate", "count"),
        average_rrp=("rrp", "mean"),
        median_rrp=("rrp", "median"),
        min_rrp=("rrp", "min"),
        max_rrp=("rrp", "max"),
        spike_intervals=("price_spike_flag", "sum"),
        average_demand_mw=("totaldemand", "mean"),
        max_demand_mw=("totaldemand", "max"),
        average_available_generation_mw=("availablegeneration", "mean"),
        average_supply_demand_gap_mw=("supply_demand_gap", "mean"),
        average_netinterchange_mw=("netinterchange", "mean"),
        average_price_change=("price_change", "mean"),
        max_price_increase=("price_change", "max"),
        max_price_decrease=("price_change", "min"),
        average_rolling_volatility_1h=("rolling_rrp_volatility_1h", "mean"),
        max_rolling_volatility_1h=("rolling_rrp_volatility_1h", "max"),
    )
    .reset_index()
)

regional_feature_summary


,regionid,intervals,average_rrp,median_rrp,min_rrp,max_rrp,spike_intervals,average_demand_mw,max_demand_mw,average_available_generation_mw,average_supply_demand_gap_mw,average_netinterchange_mw,average_price_change,max_price_increase,max_price_decrease,average_rolling_volatility_1h,max_rolling_volatility_1h
0,NSW1,8063,83.762187,65.39029,-82.96215,20300.00000,53,7872.512299,12969.33,14385.235496,6512.723197,-434.561307,1.984619e-08,19842.65742,-19827.28218,45.658856,6187.411217
1,VIC1,8063,45.055455,46.62014,-715.10671,269.77601,0,4692.430162,8786.93,10146.731392,5454.301230,654.223893,6.908993e-04,552.70748,-731.50671,9.967064,212.743180


## Step 13: Create Spike Summary By Region

### What we are doing
We are filtering intervals where RRP exceeded $300/MWh and summarising spike behaviour by region.

### Why it matters
This shows whether NSW1 or VIC1 experienced more high-price events. At this stage, we still do not explain the cause of the spikes. That explanation will come later after adding constraint and interconnector features.


In [17]:
spike_events = market_features[market_features["price_spike_flag"]].copy()

spike_summary_by_region = (
    spike_events
    .groupby("regionid")
    .agg(
        spike_intervals=("settlementdate", "count"),
        average_spike_rrp=("rrp", "mean"),
        max_spike_rrp=("rrp", "max"),
        average_spike_demand_mw=("totaldemand", "mean"),
        average_spike_supply_demand_gap_mw=("supply_demand_gap", "mean"),
        average_spike_volatility_1h=("rolling_rrp_volatility_1h", "mean"),
    )
    .reset_index()
)

spike_summary_by_region


,regionid,spike_intervals,average_spike_rrp,max_spike_rrp,average_spike_demand_mw,average_spike_supply_demand_gap_mw,average_spike_volatility_1h
0,NSW1,53,2417.098468,20300.0,11495.732075,3280.746642,1507.759777


## Check Spike Counts Including Regions With Zero Spikes

### What we are doing
The previous spike summary only shows regions that actually had spike events. If VIC1 had no intervals above $300/MWh, it will not appear in the filtered spike table.

### Why it matters
For analyst reporting, we still want to show VIC1 with zero spike intervals rather than dropping it from the summary.


In [18]:
spike_counts_all_regions = (
    market_features
    .groupby("regionid")
    .agg(
        total_intervals=("settlementdate", "count"),
        spike_intervals=("price_spike_flag", "sum"),
        max_rrp=("rrp", "max"),
        average_rrp=("rrp", "mean"),
    )
    .reset_index()
)

spike_counts_all_regions


,regionid,total_intervals,spike_intervals,max_rrp,average_rrp
0,NSW1,8063,53,20300.00000,83.762187
1,VIC1,8063,0,269.77601,45.055455


## Step 13: Create Spike Summary By Region

### What we are doing
We are summarising price spike behaviour by region, including regions that had zero spike intervals.

### Why it matters
If we only group the filtered spike event table, regions with no spikes disappear. For market reporting, that can be misleading. A region with zero spikes is still an important analytical result.

### Spike definition
A spike interval is defined as RRP greater than $300/MWh.


In [20]:
spike_summary_by_region = (
    market_features
    .groupby("regionid")
    .agg(
        total_intervals=("settlementdate", "count"),
        spike_intervals=("price_spike_flag", "sum"),
        max_rrp=("rrp", "max"),
        average_rrp=("rrp", "mean"),
        average_demand_mw=("totaldemand", "mean"),
        average_supply_demand_gap_mw=("supply_demand_gap", "mean"),
        average_volatility_1h=("rolling_rrp_volatility_1h", "mean"),
    )
    .reset_index()
)

spike_summary_by_region["spike_share_pct"] = (
    spike_summary_by_region["spike_intervals"]
    / spike_summary_by_region["total_intervals"]
    * 100
)

spike_summary_by_region


,regionid,total_intervals,spike_intervals,max_rrp,average_rrp,average_demand_mw,average_supply_demand_gap_mw,average_volatility_1h,spike_share_pct
0,NSW1,8063,53,20300.00000,83.762187,7872.512299,6512.723197,45.658856,0.657324
1,VIC1,8063,0,269.77601,45.055455,4692.430162,5454.301230,9.967064,0.000000


NSW1 recorded price spike intervals above $300/MWh during the study period, while VIC1 did not breach the selected spike threshold. This does not mean VIC1 was unimportant; it means VIC1 did not experience high-price events under this threshold. Later notebooks will still compare VIC1 against NSW1 for price divergence and interconnector-driven behaviour.


## Step 14: Preview Highest Price Intervals

### What we are doing
We are listing the highest RRP intervals across NSW1 and VIC1.

### Why it matters
This gives us a quick view of the most important high-price events in the study period. These intervals will later be tested against network constraints and interconnector flows.


In [21]:
highest_price_intervals = (
    market_features
    .sort_values("rrp", ascending=False)
    [
        [
            "settlementdate",
            "regionid",
            "rrp",
            "totaldemand",
            "availablegeneration",
            "netinterchange",
            "supply_demand_gap",
            "price_change",
            "rolling_rrp_volatility_1h",
        ]
    ]
    .head(20)
)

highest_price_intervals


,settlementdate,regionid,rrp,totaldemand,availablegeneration,netinterchange,supply_demand_gap,price_change,rolling_rrp_volatility_1h
1325,2026-02-05 14:30:00,NSW1,20300.00000,10190.15,15492.52355,589.03,5302.37355,19842.65742,5793.809073
1608,2026-02-06 14:05:00,NSW1,19036.53000,9651.93,15075.06206,-897.70,5423.13206,18882.33000,5801.771238
1368,2026-02-05 18:05:00,NSW1,9914.02372,11811.76,13709.77609,-1263.59,1898.01609,9634.29090,3719.949900
1366,2026-02-05 17:55:00,NSW1,9911.42681,11901.45,14019.02209,-1259.15,2117.57209,9608.77893,2762.209380
1350,2026-02-05 16:35:00,NSW1,9240.83000,11896.59,14384.92803,-1206.05,2488.33803,8641.47000,2519.281003
1927,2026-02-07 16:40:00,NSW1,9199.30000,11073.40,13443.29640,-946.07,2369.89640,9092.46000,2617.222174
1329,2026-02-05 14:50:00,NSW1,9199.30000,10400.88,15793.28831,-180.65,5392.40831,8801.87165,6180.726188
1602,2026-02-06 13:35:00,NSW1,9199.30000,9613.17,15758.70193,91.27,6145.53193,8899.55000,2580.566888
1327,2026-02-05 14:40:00,NSW1,7380.80815,10203.66,15819.78989,154.74,5616.12989,6908.09033,5953.998909
1372,2026-02-05 18:25:00,NSW1,1278.88000,11730.34,13132.46456,-1268.53,1402.12456,1005.22454,3665.354308


## Step 15: Save Notebook 02 Outputs

### What we are doing
We are saving the engineered market feature table and regional summaries.

### Why it matters
These files become the base inputs for later notebooks. Constraint analysis and interconnector analysis will be joined onto this feature table.


In [22]:
market_features_output = OUTPUT_DIR / "02_base_market_features.csv"
regional_summary_output = OUTPUT_DIR / "02_regional_feature_summary.csv"
spike_summary_output = OUTPUT_DIR / "02_spike_summary_by_region.csv"
highest_prices_output = OUTPUT_DIR / "02_highest_price_intervals.csv"

market_features.to_csv(market_features_output, index=False)
regional_feature_summary.to_csv(regional_summary_output, index=False)
spike_summary_by_region.to_csv(spike_summary_output, index=False)
highest_price_intervals.to_csv(highest_prices_output, index=False)

print("Saved:", market_features_output)
print("Saved:", regional_summary_output)
print("Saved:", spike_summary_output)
print("Saved:", highest_prices_output)


Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/02_base_market_features.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/02_regional_feature_summary.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/02_spike_summary_by_region.csv
Saved: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/02_highest_price_intervals.csv


## Analyst Note

Notebook 02 created the first layer of market features for NSW1 and VIC1 using the clean base market context table from Notebook 01.

The engineered features include supply-demand gap, price spike flags, price change, rolling one-hour RRP volatility, and time-based features. These features provide the market fundamentals needed before introducing network constraints and interconnector flows.

Using the $300/MWh spike threshold, NSW1 recorded price spike intervals during the February 2026 study period, while VIC1 did not breach the selected threshold. This indicates that the initial high-price event focus will be NSW1, but VIC1 remains important for later price divergence and interconnector analysis.

At this stage, spikes are only identified, not explained. The next notebook will begin analysing constraint activity to test whether network limitations contributed to high-price or divergent price outcomes.
